In [ ]:
# module_a_xor.py
# -*- coding: utf-8 -*-

"""
模块 A: XOR 实验
----------------
目标:
1. 构造 XOR 数据集
2. 用线性模型验证 XOR 线性不可分
3. 用一个两层前馈神经网络完成 XOR 分类
4. 输出训练过程与最终预测结果

说明:
- XOR 是经典的线性不可分问题
- 单层感知机/线性分类器无法正确分类
- 含隐藏层的神经网络可以学习该非线性决策边界
"""

from __future__ import annotations

import math
import random
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


def make_xor_data() -> Tuple[np.ndarray, np.ndarray]:
    """
    生成标准 XOR 数据
    X shape: (4, 2)
    y shape: (4, 1)
    """
    X = np.array([
        [0.0, 0.0],
        [0.0, 1.0],
        [1.0, 0.0],
        [1.0, 1.0],
    ], dtype=np.float64)

    y = np.array([
        [0.0],
        [1.0],
        [1.0],
        [0.0],
    ], dtype=np.float64)

    return X, y


def sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))


def sigmoid_derivative_from_output(s: np.ndarray) -> np.ndarray:
    """
    若 s = sigmoid(x)，则导数为 s * (1 - s)
    """
    return s * (1.0 - s)


@dataclass
class LinearModelResult:
    weights: np.ndarray
    bias: float
    predictions: np.ndarray
    accuracy: float


class SimpleLinearClassifier:
    """
    一个最简单的线性分类器（相当于单层 sigmoid 模型）
    用来展示 XOR 在线性模型下表现不佳。
    """

    def __init__(self, input_dim: int, lr: float = 0.1):
        self.W = np.random.randn(input_dim, 1) * 0.1
        self.b = 0.0
        self.lr = lr

    def forward(self, X: np.ndarray) -> np.ndarray:
        return sigmoid(X @ self.W + self.b)

    def fit(self, X: np.ndarray, y: np.ndarray, epochs: int = 5000) -> None:
        n = X.shape[0]

        for _ in range(epochs):
            y_hat = self.forward(X)
            error = y_hat - y

            dW = (X.T @ error) / n
            db = float(np.mean(error))

            self.W -= self.lr * dW
            self.b -= self.lr * db

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> LinearModelResult:
        prob = self.forward(X)
        pred = (prob >= 0.5).astype(np.float64)
        acc = float(np.mean(pred == y))
        return LinearModelResult(
            weights=self.W.copy(),
            bias=float(self.b),
            predictions=pred,
            accuracy=acc,
        )


class XORNeuralNetwork:
    """
    两层前馈神经网络:
        输入层(2) -> 隐藏层(hidden_dim) -> 输出层(1)

    激活函数:
        隐藏层: sigmoid
        输出层: sigmoid
    损失函数:
        均方误差 MSE
    """

    def __init__(self, input_dim: int = 2, hidden_dim: int = 4, output_dim: int = 1, lr: float = 0.5):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.lr = lr

        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.5
        self.b1 = np.zeros((1, hidden_dim), dtype=np.float64)

        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.5
        self.b2 = np.zeros((1, output_dim), dtype=np.float64)

    def forward(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        z1 = X @ self.W1 + self.b1
        a1 = sigmoid(z1)

        z2 = a1 @ self.W2 + self.b2
        a2 = sigmoid(z2)

        return a1, a2

    def compute_loss(self, y_hat: np.ndarray, y: np.ndarray) -> float:
        return float(np.mean((y_hat - y) ** 2))

    def backward(self, X: np.ndarray, y: np.ndarray, a1: np.ndarray, a2: np.ndarray) -> None:
        n = X.shape[0]

        # 输出层梯度
        delta2 = (a2 - y) * sigmoid_derivative_from_output(a2)   # (n, 1)
        dW2 = (a1.T @ delta2) / n                                # (hidden_dim, 1)
        db2 = np.mean(delta2, axis=0, keepdims=True)             # (1, 1)

        # 隐藏层梯度
        delta1 = (delta2 @ self.W2.T) * sigmoid_derivative_from_output(a1)  # (n, hidden_dim)
        dW1 = (X.T @ delta1) / n                                             # (2, hidden_dim)
        db1 = np.mean(delta1, axis=0, keepdims=True)                         # (1, hidden_dim)

        # 参数更新
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def fit(self, X: np.ndarray, y: np.ndarray, epochs: int = 10000, verbose_every: int = 1000) -> List[float]:
        loss_history: List[float] = []

        for epoch in range(1, epochs + 1):
            a1, a2 = self.forward(X)
            loss = self.compute_loss(a2, y)
            loss_history.append(loss)

            self.backward(X, y, a1, a2)

            if verbose_every > 0 and (epoch == 1 or epoch % verbose_every == 0):
                print(f"[Epoch {epoch:5d}] loss = {loss:.8f}")

        return loss_history

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        _, a2 = self.forward(X)
        return a2

    def predict(self, X: np.ndarray) -> np.ndarray:
        prob = self.predict_proba(X)
        return (prob >= 0.5).astype(np.float64)

    def accuracy(self, X: np.ndarray, y: np.ndarray) -> float:
        pred = self.predict(X)
        return float(np.mean(pred == y))


def print_dataset(X: np.ndarray, y: np.ndarray) -> None:
    print("XOR 数据集:")
    for i in range(len(X)):
        print(f"  x = {X[i].tolist()}, y = {int(y[i, 0])}")
    print()


def run_linear_baseline(X: np.ndarray, y: np.ndarray) -> None:
    print("=" * 60)
    print("一、线性模型基线")
    print("=" * 60)

    model = SimpleLinearClassifier(input_dim=2, lr=0.5)
    model.fit(X, y, epochs=5000)
    result = model.evaluate(X, y)

    print("线性模型参数:")
    print("W =")
    print(result.weights)
    print(f"b = {result.bias:.6f}")
    print()

    print("线性模型预测:")
    for xi, yi, pi in zip(X, y, result.predictions):
        print(f"  x={xi.tolist()}, 真值={int(yi[0])}, 预测={int(pi[0])}")

    print(f"\n线性模型准确率: {result.accuracy:.4f}")
    print("说明: 若准确率无法达到 1.0，说明 XOR 不是线性可分的。")
    print()


def run_xor_nn(X: np.ndarray, y: np.ndarray) -> None:
    print("=" * 60)
    print("二、两层神经网络求解 XOR")
    print("=" * 60)

    net = XORNeuralNetwork(input_dim=2, hidden_dim=4, output_dim=1, lr=1.0)
    net.fit(X, y, epochs=10000, verbose_every=1000)

    prob = net.predict_proba(X)
    pred = net.predict(X)
    acc = net.accuracy(X, y)

    print("\n神经网络预测结果:")
    for xi, yi, pr, pd in zip(X, y, prob, pred):
        print(
            f"  x={xi.tolist()}, 真值={int(yi[0])}, "
            f"预测概率={pr[0]:.6f}, 预测类别={int(pd[0])}"
        )

    print(f"\n神经网络准确率: {acc:.4f}")
    print()

    print("网络参数:")
    print("W1 =")
    print(net.W1)
    print("b1 =")
    print(net.b1)
    print("W2 =")
    print(net.W2)
    print("b2 =")
    print(net.b2)
    print()


def main() -> None:
    set_seed(42)
    X, y = make_xor_data()

    print("=" * 60)
    print("模块 A：XOR 实验")
    print("=" * 60)
    print()

    print_dataset(X, y)
    run_linear_baseline(X, y)
    run_xor_nn(X, y)

    print("=" * 60)
    print("实验结论")
    print("=" * 60)
    print("1. XOR 问题是经典的线性不可分问题。")
    print("2. 单层线性分类器无法正确表示 XOR 的判别边界。")
    print("3. 引入隐藏层后，神经网络可以通过非线性变换实现 XOR 分类。")


if __name__ == "__main__":
    main()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


def ensure_dir(path: str):
    if path is not None:
        os.makedirs(path, exist_ok=True)


def plot_xor_data(X, y, title="XOR dataset", save_path=None):
    plt.figure(figsize=(5, 5))

    y_flat = y.reshape(-1)
    class0 = y_flat == 0
    class1 = y_flat == 1

    plt.scatter(
        X[class0, 0], X[class0, 1],
        s=120, marker="o", label="Class 0"
    )
    plt.scatter(
        X[class1, 0], X[class1, 1],
        s=120, marker="^", label="Class 1"
    )

    for i in range(len(X)):
        plt.text(X[i, 0] + 0.02, X[i, 1] + 0.02, f"{i}", fontsize=10)

    plt.xlim(-0.2, 1.2)
    plt.ylim(-0.2, 1.2)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_linear_decision_boundary(model, X, y, title="Linear model decision boundary", save_path=None):
    plt.figure(figsize=(5, 5))

    y_flat = y.reshape(-1)
    class0 = y_flat == 0
    class1 = y_flat == 1

    plt.scatter(X[class0, 0], X[class0, 1], s=120, marker="o", label="Class 0")
    plt.scatter(X[class1, 0], X[class1, 1], s=120, marker="^", label="Class 1")

    w = model.W.reshape(-1)
    b = float(model.b)

    x_vals = np.linspace(-0.2, 1.2, 200)
    if abs(w[1]) > 1e-12:
        y_vals = -(w[0] * x_vals + b) / w[1]
        plt.plot(x_vals, y_vals, label="Decision boundary")
    else:
        x0 = -b / w[0]
        plt.axvline(x=x0, label="Decision boundary")

    plt.xlim(-0.2, 1.2)
    plt.ylim(-0.2, 1.2)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_nn_decision_boundary(model, X, y, title="Neural network decision boundary", save_path=None, grid_points=300):
    x_min, x_max = -0.2, 1.2
    y_min, y_max = -0.2, 1.2

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, grid_points),
        np.linspace(y_min, y_max, grid_points)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    zz = model.predict_proba(grid).reshape(xx.shape)

    plt.figure(figsize=(5, 5))
    contour = plt.contourf(xx, yy, zz, levels=30, alpha=0.6)
    plt.colorbar(contour, label="Predicted probability")

    y_flat = y.reshape(-1)
    class0 = y_flat == 0
    class1 = y_flat == 1

    plt.scatter(X[class0, 0], X[class0, 1], s=120, marker="o", edgecolors="black", label="Class 0")
    plt.scatter(X[class1, 0], X[class1, 1], s=120, marker="^", edgecolors="black", label="Class 1")

    plt.contour(xx, yy, zz, levels=[0.5], linewidths=2)

    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.legend()
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_loss_curve(loss_history, title="Training loss curve", save_path=None):
    plt.figure(figsize=(6, 4))
    plt.plot(loss_history)
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
def plot_hidden_features(model, X, y, title="Hidden-layer representation", save_path=None):
    a1, _ = model.forward(X)

    if a1.shape[1] < 2:
        raise ValueError("Hidden dimension must be at least 2 to plot hidden features.")

    plt.figure(figsize=(5, 5))

    y_flat = y.reshape(-1)
    class0 = y_flat == 0
    class1 = y_flat == 1

    plt.scatter(a1[class0, 0], a1[class0, 1], s=120, marker="o", label="Class 0")
    plt.scatter(a1[class1, 0], a1[class1, 1], s=120, marker="^", label="Class 1")

    for i in range(len(X)):
        plt.text(a1[i, 0] + 0.01, a1[i, 1] + 0.01, f"{X[i].tolist()}", fontsize=9)

    plt.xlabel("hidden unit 1")
    plt.ylabel("hidden unit 2")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# === 数据 ===
X, y = make_xor_data()

# === 线性模型 ===
linear_model = SimpleLinearClassifier(input_dim=2, lr=0.5)
linear_model.fit(X, y, epochs=5000)

# === 神经网络 ===
net = XORNeuralNetwork(input_dim=2, hidden_dim=4, output_dim=1, lr=1.0)
loss_history = net.fit(X, y, epochs=10000, verbose_every=0)

# === 绘图 ===
save_dir = "../results/figures/module_a_xor"
ensure_dir(save_dir)

plot_xor_data(X, y, save_path=f"{save_dir}/a1_xor_dataset.png")
plot_linear_decision_boundary(linear_model, X, y, save_path=f"{save_dir}/a2_linear_boundary.png")
plot_nn_decision_boundary(net, X, y, save_path=f"{save_dir}/a3_nn_boundary.png")
plot_loss_curve(loss_history, save_path=f"{save_dir}/a4_loss_curve.png")
plot_hidden_features(net, X, y, save_path=f"{save_dir}/a5_hidden_features.png")